# 🔥 Wildfire Visualization Suite
**All 4 regions | Burn severity maps | NDVI change | Fire detection maps | Grad-CAM explainability**

Run each cell top to bottom. All outputs are saved to `outputs/case_studies/<region>/visualizations/`

In [ ]:
# =========================================================
# CELL 1 — PATH SETUP (run this first, always)
# =========================================================
import os
import sys
import warnings
warnings.filterwarnings('ignore')

ROOT_DIR = r"C:\Users\vaast\OneDrive\Desktop\UAV_4_Case_Studies\wildfire-prediction-system"

if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)
if os.path.join(ROOT_DIR, 'src') not in sys.path:
    sys.path.insert(0, os.path.join(ROOT_DIR, 'src'))

os.chdir(ROOT_DIR)

from src.region_config import REGIONS
from src.spectral_indices import compute_dnbr, classify_burn_severity

print(f"✅ Working directory: {os.getcwd()}")
print(f"✅ Regions loaded: {list(REGIONS.keys())}")

In [ ]:
# =========================================================
# CELL 2 — IMPORTS & SHARED HELPERS
# =========================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import rasterio
import cv2
import torch

# ── Shared colour palettes ──────────────────────────────
REGION_COLORS = {
    'laisong':    '#E63946',
    'jyotikuchi': '#457B9D',
    'corbett':    '#2A9D8F',
    'similipal':  '#E9C46A',
}

SEVERITY_COLORS = ['#00CC00', '#99FF99', '#FFFF00', '#FFA500', '#FF4500', '#8B0000']
SEVERITY_LABELS = ['Enhanced Regrowth', 'Unburned', 'Low Severity',
                   'Moderate-Low', 'Moderate-High', 'High Severity']
SEVERITY_CMAP  = mcolors.ListedColormap(SEVERITY_COLORS)
SEVERITY_NORM  = mcolors.BoundaryNorm([-0.5,0.5,1.5,2.5,3.5,4.5,5.5], SEVERITY_CMAP.N)

# ── Utility: safe raster load ───────────────────────────
def load_band(tif_path, band_idx=1):
    """Load one band from a GeoTIFF, replace nodata with NaN."""
    with rasterio.open(tif_path) as src:
        data     = src.read(band_idx).astype(float)
        nodata   = src.nodata
        profile  = src.profile
    if nodata is not None:
        data[data == nodata] = np.nan
    data[data == -9999] = np.nan
    return data, profile

def find_tif(folder, keyword):
    """Find first .tif whose name contains keyword (case-insensitive)."""
    if not os.path.exists(folder):
        return None
    matches = [f for f in os.listdir(folder)
               if f.endswith('.tif') and keyword.lower() in f.lower()]
    return os.path.join(folder, matches[0]) if matches else None

def find_two_tifs(folder):
    """Return (earliest, latest) .tif pair from a folder, sorted by filename."""
    if not os.path.exists(folder):
        return None, None
    tifs = sorted([f for f in os.listdir(folder) if f.endswith('.tif')])
    if len(tifs) < 2:
        return None, None
    return os.path.join(folder, tifs[0]), os.path.join(folder, tifs[-1])

def ensure_dir(path):
    os.makedirs(path, exist_ok=True)
    return path

print("✅ Helpers ready.")

In [ ]:
# =========================================================
# CELL 3 — VIZ 1: BURN SEVERITY MAPS (all 4 regions)
#
# What it shows:
#   Left  → Pre-fire NBR (vegetation health before the fire)
#   Middle → Post-fire NBR (vegetation health after the fire)
#   Right  → dNBR burn severity class (USGS standard 0–5)
#
# How to read the severity colours:
#   Dark Green = vegetation grew back greener (lucky area)
#   Yellow     = low damage
#   Orange     = moderate damage
#   Dark Red   = total canopy destruction
# =========================================================

def plot_burn_severity_all_regions():
    fig = plt.figure(figsize=(22, 6 * len(REGIONS)))
    gs  = gridspec.GridSpec(len(REGIONS), 3, figure=fig,
                             hspace=0.35, wspace=0.08)

    processed = 0
    for row_idx, region_name in enumerate(REGIONS.keys()):
        idx_dir = f"data/case_studies/{region_name}/processed/indices"
        pre_path, post_path = find_two_tifs(idx_dir)

        if pre_path is None or post_path is None:
            print(f"  ⚠️  {region_name}: need ≥2 index TIFs — skipping")
            continue

        # Band 2 = NBR in our stacked (NDVI, NBR, NDMI, BSI) rasters
        nbr_pre,  _ = load_band(pre_path,  band_idx=2)
        nbr_post, _ = load_band(post_path, band_idx=2)

        dnbr     = compute_dnbr(nbr_pre, nbr_post)
        severity = classify_burn_severity(dnbr)

        pre_fname  = os.path.basename(pre_path)
        post_fname = os.path.basename(post_path)

        ax0 = fig.add_subplot(gs[row_idx, 0])
        ax1 = fig.add_subplot(gs[row_idx, 1])
        ax2 = fig.add_subplot(gs[row_idx, 2])

        # Pre-fire NBR
        im0 = ax0.imshow(nbr_pre, cmap='RdYlGn', vmin=-1, vmax=1)
        ax0.set_title(f"{REGIONS[region_name]['name']}\nPre-Fire NBR\n({pre_fname})",
                      fontsize=10, fontweight='bold')
        plt.colorbar(im0, ax=ax0, fraction=0.03, pad=0.02, label='NBR')
        ax0.axis('off')

        # Post-fire NBR
        im1 = ax1.imshow(nbr_post, cmap='RdYlGn', vmin=-1, vmax=1)
        ax1.set_title(f"Post-Fire NBR\n({post_fname})",
                      fontsize=10, fontweight='bold')
        plt.colorbar(im1, ax=ax1, fraction=0.03, pad=0.02, label='NBR')
        ax1.axis('off')

        # Burn severity
        im2 = ax2.imshow(severity, cmap=SEVERITY_CMAP, norm=SEVERITY_NORM)
        cbar = plt.colorbar(im2, ax=ax2, ticks=range(6),
                            fraction=0.03, pad=0.02)
        cbar.ax.set_yticklabels(SEVERITY_LABELS, fontsize=7)
        ax2.set_title("Burn Severity (dNBR)", fontsize=10, fontweight='bold')
        ax2.axis('off')

        # Stats annotation
        total = severity.size
        stats_lines = [f"{SEVERITY_LABELS[i]}: {(severity==i).sum()/total*100:.1f}%"
                       for i in range(6)]
        ax2.text(1.35, 0.5, '\n'.join(stats_lines),
                 transform=ax2.transAxes, fontsize=7,
                 va='center', family='monospace',
                 bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

        # Save individual region output
        out_dir = ensure_dir(f"outputs/case_studies/{region_name}/visualizations")
        fig_single, axes_s = plt.subplots(1, 3, figsize=(18, 5))
        axes_s[0].imshow(nbr_pre,  cmap='RdYlGn', vmin=-1, vmax=1); axes_s[0].axis('off')
        axes_s[0].set_title(f"Pre-Fire NBR\n{pre_fname}", fontweight='bold')
        axes_s[1].imshow(nbr_post, cmap='RdYlGn', vmin=-1, vmax=1); axes_s[1].axis('off')
        axes_s[1].set_title(f"Post-Fire NBR\n{post_fname}", fontweight='bold')
        im_s = axes_s[2].imshow(severity, cmap=SEVERITY_CMAP, norm=SEVERITY_NORM)
        cb_s = plt.colorbar(im_s, ax=axes_s[2], ticks=range(6))
        cb_s.ax.set_yticklabels(SEVERITY_LABELS, fontsize=8)
        axes_s[2].axis('off')
        axes_s[2].set_title("Burn Severity", fontweight='bold')
        fig_single.suptitle(f"{REGIONS[region_name]['name']} — Burn Severity Map",
                             fontsize=13, fontweight='bold')
        plt.tight_layout()
        fig_single.savefig(f"{out_dir}/burn_severity_map.png", dpi=250, bbox_inches='tight')
        plt.close(fig_single)

        processed += 1
        print(f"  ✅ {REGIONS[region_name]['name']} — severity map done")

    fig.suptitle("Burn Severity Maps — All Regions (Pre vs Post Fire NBR)",
                 fontsize=15, fontweight='bold', y=1.002)
    out_dir_all = ensure_dir("outputs/comparative_analysis")
    fig.savefig(f"{out_dir_all}/ALL_burn_severity_maps.png",
                dpi=200, bbox_inches='tight')
    plt.show()
    print(f"\n✅ Combined map saved → outputs/comparative_analysis/ALL_burn_severity_maps.png")


plot_burn_severity_all_regions()

In [ ]:
# =========================================================
# CELL 4 — VIZ 2: NDVI CHANGE MAPS (vegetation loss)
#
# NDVI = how green/healthy the vegetation is (-1 to +1)
# A drop in NDVI after fire = vegetation was burned
#
# Red  → vegetation was lost (bad)
# Blue → vegetation grew or stayed healthy
# =========================================================

def plot_ndvi_change_all_regions():
    fig, axes = plt.subplots(len(REGIONS), 3,
                              figsize=(20, 5.5 * len(REGIONS)))
    if len(REGIONS) == 1:
        axes = [axes]

    for row_idx, region_name in enumerate(REGIONS.keys()):
        idx_dir = f"data/case_studies/{region_name}/processed/indices"
        pre_path, post_path = find_two_tifs(idx_dir)

        ax_pre, ax_post, ax_diff = axes[row_idx]

        if pre_path is None or post_path is None:
            for ax in [ax_pre, ax_post, ax_diff]:
                ax.text(0.5, 0.5, 'No TIF data', ha='center', va='center',
                        transform=ax.transAxes, fontsize=12)
                ax.axis('off')
            continue

        # Band 1 = NDVI in stacked raster
        ndvi_pre,  _ = load_band(pre_path,  band_idx=1)
        ndvi_post, _ = load_band(post_path, band_idx=1)
        ndvi_diff    = ndvi_post - ndvi_pre   # negative = vegetation lost

        region_label = REGIONS[region_name]['name']

        im0 = ax_pre.imshow(ndvi_pre,  cmap='RdYlGn', vmin=-0.5, vmax=0.9)
        ax_pre.set_title(f"{region_label}\nNDVI Pre-Fire",
                         fontsize=10, fontweight='bold')
        plt.colorbar(im0, ax=ax_pre, fraction=0.03, label='NDVI')
        ax_pre.axis('off')

        im1 = ax_post.imshow(ndvi_post, cmap='RdYlGn', vmin=-0.5, vmax=0.9)
        ax_post.set_title("NDVI Post-Fire", fontsize=10, fontweight='bold')
        plt.colorbar(im1, ax=ax_post, fraction=0.03, label='NDVI')
        ax_post.axis('off')

        # Symmetric diverging colormap: red = loss, blue = gain
        vmax = max(abs(np.nanpercentile(ndvi_diff, 2)),
                   abs(np.nanpercentile(ndvi_diff, 98)))
        im2 = ax_diff.imshow(ndvi_diff, cmap='RdBu', vmin=-vmax, vmax=vmax)
        ax_diff.set_title("ΔNDVI (Post − Pre)\nRed=Loss  Blue=Gain",
                          fontsize=10, fontweight='bold')
        plt.colorbar(im2, ax=ax_diff, fraction=0.03, label='ΔNDVI')
        ax_diff.axis('off')

        # Annotate mean loss
        mean_loss = float(np.nanmean(ndvi_diff))
        pct_lost  = float(np.nanmean(ndvi_diff < -0.1) * 100)
        ax_diff.text(0.02, 0.96,
                     f"Mean ΔNDVI: {mean_loss:+.3f}\n{pct_lost:.1f}% pixels lost >0.1",
                     transform=ax_diff.transAxes, fontsize=8,
                     va='top', color='black',
                     bbox=dict(boxstyle='round', facecolor='white', alpha=0.75))

        # Save individual
        out_dir = ensure_dir(f"outputs/case_studies/{region_name}/visualizations")
        fig_s, axs = plt.subplots(1, 3, figsize=(18, 5))
        axs[0].imshow(ndvi_pre,  cmap='RdYlGn', vmin=-0.5, vmax=0.9)
        axs[0].set_title('NDVI Pre', fontweight='bold'); axs[0].axis('off')
        axs[1].imshow(ndvi_post, cmap='RdYlGn', vmin=-0.5, vmax=0.9)
        axs[1].set_title('NDVI Post', fontweight='bold'); axs[1].axis('off')
        axs[2].imshow(ndvi_diff, cmap='RdBu', vmin=-vmax, vmax=vmax)
        axs[2].set_title('ΔNDVI', fontweight='bold'); axs[2].axis('off')
        fig_s.suptitle(f"{region_label} — NDVI Change Map",
                       fontsize=13, fontweight='bold')
        plt.tight_layout()
        fig_s.savefig(f"{out_dir}/ndvi_change_map.png", dpi=250, bbox_inches='tight')
        plt.close(fig_s)
        print(f"  ✅ {region_label} — NDVI change map done")

    fig.suptitle("NDVI Vegetation Change Maps — All Regions",
                 fontsize=15, fontweight='bold')
    plt.tight_layout()
    fig.savefig("outputs/comparative_analysis/ALL_ndvi_change_maps.png",
                dpi=200, bbox_inches='tight')
    plt.show()
    print("\n✅ Combined NDVI map saved.")


plot_ndvi_change_all_regions()

In [ ]:
# =========================================================
# CELL 5 — VIZ 3: FIRE DETECTION POINT MAPS
#
# Plots every NASA FIRMS satellite fire detection as a dot
# on the region map, coloured by:
#   - Colour = year (so you can see which years were bad)
#   - Size   = Fire Radiative Power (bigger dot = hotter fire)
# =========================================================

import matplotlib.cm as cm

def plot_fire_detection_maps():
    fig, axes = plt.subplots(2, 2, figsize=(18, 14))
    axes = axes.flatten()

    for idx, region_name in enumerate(REGIONS.keys()):
        ax  = axes[idx]
        cfg = REGIONS[region_name]
        csv = f"data/case_studies/{region_name}/raw/firms/firms_{region_name}.csv"

        if not os.path.exists(csv):
            ax.text(0.5, 0.5, 'No FIRMS data', ha='center', va='center',
                    transform=ax.transAxes, fontsize=12)
            ax.set_title(cfg['name'])
            continue

        df = pd.read_csv(csv)
        df['acq_date'] = pd.to_datetime(df['acq_date'])
        df['year']     = df['acq_date'].dt.year

        years     = sorted(df['year'].unique())
        cmap_year = cm.get_cmap('plasma', len(years))
        year_to_c = {yr: cmap_year(i) for i, yr in enumerate(years)}

        # Marker size proportional to FRP (clipped so outliers don't dominate)
        frp_clipped = df['frp'].clip(upper=df['frp'].quantile(0.95))
        sizes       = 5 + (frp_clipped / frp_clipped.max()) * 60

        for yr in years:
            mask = df['year'] == yr
            ax.scatter(
                df.loc[mask, 'longitude'],
                df.loc[mask, 'latitude'],
                c=[year_to_c[yr]],
                s=sizes[mask],
                alpha=0.45,
                linewidths=0,
                label=str(yr),
            )

        # Bounding box
        b = cfg['bounds']
        rect = plt.Rectangle(
            (b['lon_min'], b['lat_min']),
            b['lon_max'] - b['lon_min'],
            b['lat_max'] - b['lat_min'],
            linewidth=2, edgecolor='navy', facecolor='none', linestyle='--'
        )
        ax.add_patch(rect)

        # Centre marker
        ax.plot(cfg['center']['lon'], cfg['center']['lat'],
                'k^', markersize=9, zorder=5, label='Centre')

        # Size legend
        for frp_val, lbl in [(5, 'Low FRP'), (50, 'Mid FRP'), (100, 'High FRP')]:
            ax.scatter([], [], c='grey',
                       s=5 + (frp_val / frp_clipped.max()) * 60,
                       label=f'{lbl} (~{frp_val} MW)', alpha=0.6)

        ax.set_title(
            f"{cfg['name']}\n"
            f"{len(df):,} detections | {df['year'].min()}–{df['year'].max()}",
            fontsize=10, fontweight='bold'
        )
        ax.set_xlabel('Longitude')
        ax.set_ylabel('Latitude')
        ax.set_xlim(b['lon_min'] - 0.05, b['lon_max'] + 0.05)
        ax.set_ylim(b['lat_min'] - 0.05, b['lat_max'] + 0.05)
        ax.legend(fontsize=6, loc='lower right', ncol=2,
                  markerscale=1.2, framealpha=0.8)
        ax.grid(alpha=0.25)

        out_dir = ensure_dir(f"outputs/case_studies/{region_name}/visualizations")
        print(f"  ✅ {cfg['name']} — detection map done")

    fig.suptitle(
        "NASA FIRMS Fire Detection Maps — All Regions\n"
        "(Colour = year, Dot size = fire intensity)",
        fontsize=14, fontweight='bold'
    )
    plt.tight_layout()
    fig.savefig("outputs/comparative_analysis/ALL_fire_detection_maps.png",
                dpi=250, bbox_inches='tight')
    plt.show()
    print("\n✅ Fire detection maps saved.")


plot_fire_detection_maps()

In [ ]:
# =========================================================
# CELL 6 — VIZ 4: SPECTRAL INDEX COMPARISON DASHBOARD
#
# Side-by-side view of NDVI, NBR, NDMI, BSI for each region
# from the most recent satellite composite available.
#
# NDVI = vegetation greenness
# NBR  = burn scar detection
# NDMI = moisture content
# BSI  = bare soil exposure
# =========================================================

INDEX_BANDS = {
    'NDVI': {'band': 1, 'cmap': 'RdYlGn',  'vmin': -0.3, 'vmax': 0.9},
    'NBR':  {'band': 2, 'cmap': 'RdYlGn',  'vmin': -0.5, 'vmax': 0.8},
    'NDMI': {'band': 3, 'cmap': 'RdYlBu',  'vmin': -0.5, 'vmax': 0.5},
    'BSI':  {'band': 4, 'cmap': 'YlOrBr',  'vmin': -0.5, 'vmax': 0.5},
}

def plot_spectral_index_dashboard():
    n_regions = len(REGIONS)
    n_indices = len(INDEX_BANDS)

    fig, axes = plt.subplots(n_regions, n_indices,
                              figsize=(5 * n_indices, 4.5 * n_regions))
    if n_regions == 1:
        axes = [axes]

    for row_idx, region_name in enumerate(REGIONS.keys()):
        idx_dir  = f"data/case_studies/{region_name}/processed/indices"
        tifs = sorted([f for f in os.listdir(idx_dir) if f.endswith('.tif')]) \
               if os.path.exists(idx_dir) else []
        latest_tif = os.path.join(idx_dir, tifs[-1]) if tifs else None

        for col_idx, (idx_name, cfg) in enumerate(INDEX_BANDS.items()):
            ax = axes[row_idx][col_idx]

            if latest_tif is None:
                ax.text(0.5, 0.5, 'No TIF', ha='center', va='center',
                        transform=ax.transAxes)
                ax.axis('off')
                continue

            try:
                data, _ = load_band(latest_tif, band_idx=cfg['band'])
            except Exception:
                ax.text(0.5, 0.5, f'Band {cfg["band"]} missing',
                        ha='center', va='center', transform=ax.transAxes)
                ax.axis('off')
                continue

            im = ax.imshow(data, cmap=cfg['cmap'],
                           vmin=cfg['vmin'], vmax=cfg['vmax'])
            plt.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
            ax.axis('off')

            title_region = REGIONS[region_name]['name'] if col_idx == 0 else ''
            ax.set_title(
                f"{title_region}\n{idx_name}" if title_region else idx_name,
                fontsize=9, fontweight='bold'
            )

            # Stats
            mean_v = float(np.nanmean(data))
            ax.text(0.02, 0.03, f"mean={mean_v:.3f}",
                    transform=ax.transAxes, fontsize=7,
                    bbox=dict(facecolor='white', alpha=0.7))

        print(f"  ✅ {REGIONS[region_name]['name']} — spectral dashboard done")

    fig.suptitle("Spectral Index Dashboard — All Regions (Latest Composite)",
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    fig.savefig("outputs/comparative_analysis/ALL_spectral_index_dashboard.png",
                dpi=200, bbox_inches='tight')
    plt.show()
    print("\n✅ Spectral index dashboard saved.")


plot_spectral_index_dashboard()

In [ ]:
# =========================================================
# CELL 7 — VIZ 5: SEASONAL FIRE HEATMAPS
#
# A heatmap grid of Year (rows) × Month (columns) for each region.
# Darker red cell = more fires that month & year.
# Instantly shows which years were unusually bad and in which month.
# =========================================================

import seaborn as sns

def plot_seasonal_heatmaps():
    fig, axes = plt.subplots(2, 2, figsize=(20, 12))
    axes = axes.flatten()
    month_labels = ['Jan','Feb','Mar','Apr','May','Jun',
                    'Jul','Aug','Sep','Oct','Nov','Dec']

    for idx, region_name in enumerate(REGIONS.keys()):
        ax  = axes[idx]
        csv = f"data/case_studies/{region_name}/raw/firms/firms_{region_name}.csv"

        if not os.path.exists(csv):
            ax.text(0.5, 0.5, 'No data', ha='center', va='center',
                    transform=ax.transAxes)
            continue

        df = pd.read_csv(csv)
        df['acq_date'] = pd.to_datetime(df['acq_date'])
        df['year']  = df['acq_date'].dt.year
        df['month'] = df['acq_date'].dt.month

        pivot = df.groupby(['year','month']).size().unstack(fill_value=0)
        pivot.columns = [month_labels[m-1] for m in pivot.columns]

        # Reindex to make sure all 12 months are present
        for ml in month_labels:
            if ml not in pivot.columns:
                pivot[ml] = 0
        pivot = pivot[month_labels]

        sns.heatmap(
            pivot, ax=ax,
            cmap='YlOrRd',
            linewidths=0.5, linecolor='white',
            annot=True, fmt='d', annot_kws={'size': 7},
            cbar_kws={'label': 'Detections'},
        )
        ax.set_title(f"{REGIONS[region_name]['name']}\nYear × Month Fire Count",
                     fontsize=11, fontweight='bold')
        ax.set_xlabel('Month')
        ax.set_ylabel('Year')
        ax.tick_params(axis='x', rotation=30)

        out_dir = ensure_dir(f"outputs/case_studies/{region_name}/visualizations")
        fig_s, ax_s = plt.subplots(figsize=(13, 5))
        sns.heatmap(pivot, ax=ax_s, cmap='YlOrRd', linewidths=0.5,
                    linecolor='white', annot=True, fmt='d',
                    annot_kws={'size': 8}, cbar_kws={'label': 'Detections'})
        ax_s.set_title(f"{REGIONS[region_name]['name']} — Seasonal Fire Heatmap",
                       fontsize=12, fontweight='bold')
        ax_s.tick_params(axis='x', rotation=30)
        fig_s.tight_layout()
        fig_s.savefig(f"{out_dir}/seasonal_heatmap.png", dpi=250, bbox_inches='tight')
        plt.close(fig_s)
        print(f"  ✅ {REGIONS[region_name]['name']} — seasonal heatmap done")

    fig.suptitle("Seasonal Fire Detection Heatmaps — All Regions",
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    fig.savefig("outputs/comparative_analysis/ALL_seasonal_heatmaps.png",
                dpi=200, bbox_inches='tight')
    plt.show()
    print("\n✅ Seasonal heatmaps saved.")


plot_seasonal_heatmaps()

In [ ]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import timm
from src.region_config import REGIONS

# =========================================================
# 1. LOCAL MODEL DEFINITION (Bypasses Windows Security Block)
# =========================================================
class WildfireCNN(nn.Module):
    def __init__(self, in_channels=5, pretrained=True, n_train_tiles=9999):
        super().__init__()
        if n_train_tiles < 300:
            backbone_name = 'efficientnet_b0'
        elif n_train_tiles < 1500:
            backbone_name = 'efficientnet_b1'
        else:
            backbone_name = 'efficientnet_b3'

        self.backbone = timm.create_model(
            backbone_name, pretrained=pretrained, num_classes=0, in_chans=in_channels
        )
        n_features = self.backbone.num_features
        dropout_head = 0.5 if n_train_tiles < 500 else 0.4
        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout_head),
            nn.Linear(n_features, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(256, 1),
        )

    def forward(self, x):
        return self.classifier(self.backbone(x)).squeeze(1)


# =========================================================
# 2. GRAD-CAM EXPLAINABILITY LOGIC
# =========================================================
class GradCAM:
    """Gradient-weighted Class Activation Mapping for WildfireCNN."""
    def __init__(self, model):
        self.model       = model
        self.gradients   = None
        self.activations = None

        target_layer = model.backbone.blocks[-1]

        def save_activation(module, inp, out):
            self.activations = out
            out.register_hook(lambda g: setattr(self, 'gradients', g))

        target_layer.register_forward_hook(save_activation)

    def generate(self, tile_tensor):
        self.model.eval()
        x      = tile_tensor.unsqueeze(0)   # → (1, C, H, W)
        logit  = self.model(x)
        prob   = float(torch.sigmoid(logit).item())

        self.model.zero_grad()
        logit.backward()

        grads = self.gradients[0]       # (C, H, W)
        acts  = self.activations[0]     # (C, H, W)
        weights = grads.mean(dim=(1, 2))
        cam     = (weights[:, None, None] * acts).sum(dim=0)
        cam     = torch.relu(cam).detach().cpu().numpy()
        cam     = cv2.resize(cam, (256, 256))

        lo, hi = cam.min(), cam.max()
        cam    = (cam - lo) / (hi - lo + 1e-8)
        return cam, prob


def ensure_dir(path):
    os.makedirs(path, exist_ok=True)
    return path


def gradcam_for_region(region_name, device):
    """Run Grad-CAM on one fire tile + one no-fire tile for a region."""
    model_path = f"outputs/case_studies/{region_name}/models/cnn_best.pt"
    tile_dir   = f"data/case_studies/{region_name}/processed/tiles"

    if not os.path.exists(model_path):
        print(f"  ⚠️  {region_name}: model not found at {model_path}")
        return

    n_fire = len([f for f in os.listdir(os.path.join(tile_dir, 'pre_fire')) if f.endswith('.npy')]) if os.path.exists(os.path.join(tile_dir, 'pre_fire')) else 0
    n_nofire = len([f for f in os.listdir(os.path.join(tile_dir, 'no_fire')) if f.endswith('.npy')]) if os.path.exists(os.path.join(tile_dir, 'no_fire')) else 0
    n_train = int((n_fire + n_nofire) * 0.85)

    model = WildfireCNN(in_channels=5, pretrained=False, n_train_tiles=n_train).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    gradcam = GradCAM(model)

    examples = []
    for subdir, label in [('pre_fire', 'Fire'), ('no_fire', 'No Fire')]:
        folder = os.path.join(tile_dir, subdir)
        if not os.path.exists(folder): continue
        tiles = [f for f in os.listdir(folder) if f.endswith('.npy')]
        if not tiles: continue
        examples.append((os.path.join(folder, tiles[0]), label))

    if not examples:
        return

    fig, axes = plt.subplots(len(examples), 3, figsize=(13, 4.5 * len(examples)))
    if len(examples) == 1: axes = [axes]

    for row_i, (tile_path, label) in enumerate(examples):
        tile_np = np.load(tile_path).astype(np.float32)

        for c in range(tile_np.shape[0]):
            tile_np[c] = (tile_np[c] - tile_np[c].mean()) / (tile_np[c].std() + 1e-6)

        if tile_np.shape[0] > 5:
            tile_np = tile_np[:5]
        elif tile_np.shape[0] < 5:
            pad = np.zeros((5-tile_np.shape[0], *tile_np.shape[1:]), dtype=np.float32)
            tile_np = np.concatenate([tile_np, pad], axis=0)

        tile_t = torch.tensor(tile_np, dtype=torch.float32).to(device)
        cam, prob = gradcam.generate(tile_t)

        rgb = tile_np[:3].transpose(1, 2, 0)
        rgb = ((rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-8) * 255).astype(np.uint8)

        heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
        heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
        overlay = cv2.addWeighted(rgb, 0.55, heatmap, 0.45, 0)

        ax0, ax1, ax2 = axes[row_i]
        ax0.imshow(rgb)
        ax0.set_title(f"[{label}] RGB tile", fontweight='bold')
        ax0.axis('off')

        ax1.imshow(cam, cmap='jet')
        ax1.set_title(f"Grad-CAM heatmap\nP(fire)={prob:.3f}", fontweight='bold')
        ax1.axis('off')

        ax2.imshow(overlay)
        ax2.set_title("Overlay", fontweight='bold')
        ax2.axis('off')

    fig.suptitle(
        f"CNN Explainability (Grad-CAM) — {REGIONS[region_name]['name']}\n"
        "Red = where the model looked | Blue = ignored",
        fontsize=12, fontweight='bold'
    )
    plt.tight_layout()
    out_dir = ensure_dir(f"outputs/case_studies/{region_name}/visualizations")
    fig.savefig(f"{out_dir}/gradcam_fire_vs_nofire.png", dpi=200, bbox_inches='tight')
    plt.show()
    print(f"  ✅ {REGIONS[region_name]['name']} — Grad-CAM saved")


# =========================================================
# 3. EXECUTION
# =========================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️  Device: {device}\n")

for region_name in REGIONS.keys():
    print(f"\n🔍 Grad-CAM: {REGIONS[region_name]['name']}")
    gradcam_for_region(region_name, device)

In [ ]:
# =========================================================
# CELL 9 — SUMMARY: list all saved files
# =========================================================

print("\n" + "="*60)
print("✅ ALL VISUALIZATIONS COMPLETE")
print("="*60)

print("\n📁 Per-region outputs (outputs/case_studies/<region>/visualizations/):")
for rn in REGIONS.keys():
    out_dir = f"outputs/case_studies/{rn}/visualizations"
    if os.path.exists(out_dir):
        files = os.listdir(out_dir)
        print(f"  {REGIONS[rn]['name']}:")
        for f in sorted(files):
            print(f"    • {f}")

print("\n📁 Combined outputs (outputs/comparative_analysis/):")
comp_dir = "outputs/comparative_analysis"
if os.path.exists(comp_dir):
    for f in sorted(os.listdir(comp_dir)):
        if f.startswith('ALL_'):
            print(f"  • {f}")